# M7 OOS Validation
**时间切分**: Train 70% / Val 15% / Test 15%  
**铁律**: 调参只看 Train+Val；Test 集**一次性**最终评估，结果不可回改参数。

In [1]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from spx_scanner.data_layer.loader import load_data
from spx_scanner.data_layer.resampler import resample_1m_to_3m
from spx_scanner.features import compute_all_features
from spx_scanner.scanner.engine import ScannerEngine
from spx_scanner.patterns.registry import get_all_patterns
from spx_scanner.backtest.simulator import run_backtest
from spx_scanner.backtest.metrics import (
    compute_metrics, confidence_calibration, learning_curve
)
from spx_scanner.backtest.grouping import enrich_trades, session_heatmap
from spx_scanner.config_loader import load_config

cfg = load_config()
TRAIN_PCT = cfg['split']['train_pct']
VAL_PCT   = cfg['split']['val_pct']
TEST_PCT  = cfg['split']['test_pct']
MIN_SAMPLES = cfg['backtest']['min_samples_per_pattern']

print(f'Split: Train={TRAIN_PCT:.0%} / Val={VAL_PCT:.0%} / Test={TEST_PCT:.0%}')
print(f'Min samples per pattern: {MIN_SAMPLES}')

Split: Train=70% / Val=15% / Test=15%
Min samples per pattern: 30


In [2]:
# ── 加载数据 & 特征 ────────────────────────────────────────────────────────
df1m = load_data('../data/spy_1min.parquet')
df3m = resample_1m_to_3m(df1m)
df   = compute_all_features(df3m)

dates = sorted(df.index.normalize().unique())
n_days = len(dates)

n_train = int(n_days * TRAIN_PCT)
n_val   = int(n_days * VAL_PCT)
n_test  = n_days - n_train - n_val

train_dates = dates[:n_train]
val_dates   = dates[n_train:n_train + n_val]
test_dates  = dates[n_train + n_val:]

df_train = df[df.index.normalize().isin(train_dates)]
df_val   = df[df.index.normalize().isin(val_dates)]
df_test  = df[df.index.normalize().isin(test_dates)]

print(f'Total trading days: {n_days}')
print(f'Train: {n_train} days ({train_dates[0].date()} ~ {train_dates[-1].date()}) — {len(df_train)} bars')
print(f'Val:   {n_val} days ({val_dates[0].date()} ~ {val_dates[-1].date()}) — {len(df_val)} bars')
print(f'Test:  {n_test} days ({test_dates[0].date()} ~ {test_dates[-1].date()}) — {len(df_test)} bars')

Total trading days: 14
Train: 9 days (2026-04-06 ~ 2026-04-16) — 1170 bars
Val:   2 days (2026-04-17 ~ 2026-04-20) — 260 bars
Test:  3 days (2026-04-21 ~ 2026-04-23) — 390 bars


In [3]:
# ── 扫描 & 回测 (各 split) ─────────────────────────────────────────────────
patterns = get_all_patterns('SPY')
engine   = ScannerEngine(patterns=patterns)
STRATEGY = 'target_stop'  # M7 主要看 target_stop

results = {}
for split_name, split_df in [('train', df_train), ('val', df_val), ('test', df_test)]:
    sigs = engine.scan(split_df)
    trades = run_backtest(split_df, sigs)
    if not trades.empty:
        trades = trades[trades['exit_strategy'] == STRATEGY]
    results[split_name] = {'sigs': sigs, 'trades': trades}
    n_sigs = len(sigs)
    n_tr   = len(trades)
    wr = float((trades['pnl_pct'] > 0).mean()) if not trades.empty else float('nan')
    print(f'[{split_name:5s}] signals={n_sigs:4d}  trades={n_tr:4d}  win_rate={wr:.1%}')

[train] signals= 194  trades= 194  win_rate=58.2%
[val  ] signals=  38  trades=  38  win_rate=31.6%
[test ] signals=  58  trades=  58  win_rate=36.2%


In [4]:
# ── Per-split Per-pattern 指标表 ──────────────────────────────────────────
def _metrics_for(trades, split_name):
    if trades.empty:
        return pd.DataFrame()
    m = compute_metrics(trades.assign(exit_strategy=STRATEGY))
    m['split'] = split_name
    return m

all_metrics = pd.concat([
    _metrics_for(results[s]['trades'], s)
    for s in ['train', 'val', 'test']
    if not results[s]['trades'].empty
])

if not all_metrics.empty:
    all_metrics = all_metrics.reset_index()
    pivot = all_metrics.pivot_table(
        index='pattern', columns='split',
        values=['n_trades', 'win_rate', 'expect_pct', 'profit_factor', 'max_drawdown_pct'],
        aggfunc='first'
    )
    print(pivot.to_string())

                expect_pct                 max_drawdown_pct                 n_trades             profit_factor               win_rate              
split                 test   train     val             test   train     val     test train   val          test  train    val     test  train    val
pattern                                                                                                                                            
failed_breakout    -0.3419  0.0182     NaN          -0.8100 -0.1652     NaN      2.0  18.0   NaN         0.156  1.652    NaN    0.500  0.444    NaN
last_hour_drift    -0.1053  0.0314 -0.0682          -1.3676 -1.0217 -0.9403     15.0  82.0  15.0         0.154  2.279  0.001    0.267  0.707  0.067
liquidity_sweep     0.0820 -0.0542  0.0806          -0.1156 -0.5918  0.0000      4.0  10.0   2.0         2.646  0.401  9.440    0.500  0.400  0.500
orb_breakout       -0.0900  0.0131 -0.0216          -3.2117 -2.3288 -0.7831     36.0  74.0  21.0         0.443  

In [5]:
# ── 统计有效性检验 ─────────────────────────────────────────────────────────
print('=== Statistical Validity Check (Test Set) ===')
print(f'Minimum required samples per pattern: {MIN_SAMPLES}')
print()

test_trades = results['test']['trades']
valid_patterns = []

if not test_trades.empty:
    for pat, grp in test_trades.groupby('pattern'):
        n = len(grp)
        wr = float((grp['pnl_pct'] > 0).mean())
        exp = float(grp['pnl_pct'].mean())

        # 二项检验:Win Rate vs 50% (H0: WR = 0.5)
        from scipy import stats as sp_stats
        wins = int((grp['pnl_pct'] > 0).sum())
        pval = sp_stats.binomtest(wins, n, p=0.5).pvalue

        valid = n >= MIN_SAMPLES
        status = '✅' if valid else '⚠️ (insufficient)'
        sig    = '* p<0.05' if pval < 0.05 else ''
        print(f'  {pat:25s}: n={n:3d}  WR={wr:.1%}  E[PnL]={exp:+.3f}%  p={pval:.3f} {sig}  {status}')
        if valid:
            valid_patterns.append(pat)

print(f'\nStatistically valid patterns: {valid_patterns if valid_patterns else "None (need more data)"}')

=== Statistical Validity Check (Test Set) ===
Minimum required samples per pattern: 30

  failed_breakout          : n=  2  WR=50.0%  E[PnL]=-0.342%  p=1.000   ⚠️ (insufficient)
  last_hour_drift          : n= 15  WR=26.7%  E[PnL]=-0.105%  p=0.118   ⚠️ (insufficient)
  liquidity_sweep          : n=  4  WR=50.0%  E[PnL]=+0.082%  p=1.000   ⚠️ (insufficient)
  orb_breakout             : n= 36  WR=38.9%  E[PnL]=-0.090%  p=0.243   ✅
  vwap_rejection           : n=  1  WR=0.0%  E[PnL]=-0.120%  p=1.000   ⚠️ (insufficient)

Statistically valid patterns: ['orb_breakout']


In [6]:
# ── Train+Val 置信度校准 (调参参考) ──────────────────────────────────────
trainval_trades = pd.concat([
    results['train']['trades'],
    results['val']['trades'],
], ignore_index=True)

if not trainval_trades.empty:
    calib_tv = confidence_calibration(trainval_trades, n_bins=5)
    print('=== Confidence Calibration (Train+Val) ===')
    print(calib_tv.to_string())
    print()

    # 是否单调递增?
    wr_vals = calib_tv['actual_win_rate'].dropna().values
    is_monotone = all(wr_vals[i] <= wr_vals[i+1] for i in range(len(wr_vals)-1))
    print(f'Confidence → Win Rate monotone increasing: {is_monotone}')
    if not is_monotone:
        print('  → 置信度校准存在问题,考虑重新设计 confidence scoring')

=== Confidence Calibration (Train+Val) ===
   bin_low  bin_high  n_trades  actual_win_rate  sufficient
0     0.50      0.58         8            0.125        True
1     0.58      0.66        54            0.574        True
2     0.66      0.74       129            0.574        True
3     0.74      0.82        39            0.487        True
4     0.82      0.90         2            0.000       False

Confidence → Win Rate monotone increasing: False
  → 置信度校准存在问题,考虑重新设计 confidence scoring


In [7]:
# ── 调参探索 (仅 Train+Val) ──────────────────────────────────────────────
# 探索 confidence threshold 对 Win Rate 的影响
print('=== Confidence Threshold Sweep (Train+Val only) ===')
print(f'{"Conf_min":>10} {"n_trades":>10} {"win_rate":>10} {"expect_pct":>12} {"profit_factor":>14}')

for conf_min in [0.50, 0.55, 0.60, 0.65, 0.70]:
    subset = trainval_trades[trainval_trades['confidence'] >= conf_min]
    if subset.empty:
        continue
    n   = len(subset)
    wr  = float((subset['pnl_pct'] > 0).mean())
    exp = float(subset['pnl_pct'].mean())
    wins_sum  = float(subset.loc[subset['pnl_pct']>0, 'pnl_pct'].sum())
    loss_sum  = float(subset.loc[subset['pnl_pct']<0, 'pnl_pct'].abs().sum())
    pf = wins_sum / loss_sum if loss_sum > 0 else float('inf')
    print(f'{conf_min:>10.2f} {n:>10d} {wr:>10.1%} {exp:>12.4f}% {pf:>14.3f}')

=== Confidence Threshold Sweep (Train+Val only) ===
  Conf_min   n_trades   win_rate   expect_pct  profit_factor
      0.50        232      53.9%       0.0106%          1.216
      0.55        224      55.4%       0.0151%          1.327
      0.60        224      55.4%       0.0151%          1.327
      0.65        175      54.9%       0.0175%          1.520
      0.70        170      54.7%       0.0182%          1.545


In [8]:
# ── 最终 OOS 评估 (Test Set — 只运行一次) ──────────────────────────────────
print('=' * 60)
print('FINAL OUT-OF-SAMPLE EVALUATION (TEST SET)')
print('This result must NOT be used to tune parameters.')
print('=' * 60)

if not test_trades.empty:
    final = compute_metrics(test_trades.assign(exit_strategy=STRATEGY))
    cols  = ['n_trades','win_rate','expect_pct','profit_factor','sharpe','max_drawdown_pct','calmar']
    print(final[[c for c in cols if c in final.columns]].to_string())
    
    overall_wr  = float((test_trades['pnl_pct'] > 0).mean())
    overall_exp = float(test_trades['pnl_pct'].mean())
    print(f'\nOverall Test: n={len(test_trades)}  WR={overall_wr:.1%}  E[PnL]={overall_exp:+.4f}%')
else:
    print('No trades in test set.')

FINAL OUT-OF-SAMPLE EVALUATION (TEST SET)
This result must NOT be used to tune parameters.
                               n_trades  win_rate  expect_pct  profit_factor  sharpe  max_drawdown_pct    calmar
pattern         exit_strategy                                                                                   
failed_breakout target_stop           2     0.500     -0.3419          0.156  -0.730           -0.8100 -2286.931
last_hour_drift target_stop          15     0.267     -0.1053          0.154  -2.970           -1.3676   -55.601
liquidity_sweep target_stop           4     0.500      0.0820          2.646   0.553           -0.1156  1922.195
orb_breakout    target_stop          36     0.389     -0.0900          0.443  -1.787           -3.2117    -8.437
vwap_rejection  target_stop           1     0.000     -0.1200          0.000   0.000            0.0000       NaN

Overall Test: n=58  WR=36.2%  E[PnL]=-0.0913%


In [9]:
# ── OOS 综合图表 ───────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle('M7 OOS Validation — SPY 3min', fontsize=14, fontweight='bold')

SPLIT_COLORS = {'train': '#4CAF50', 'val': '#2196F3', 'test': '#FF5722'}

# 1. Win Rate 对比 (Train/Val/Test per pattern)
ax1 = fig.add_subplot(gs[0, 0])
if not all_metrics.empty:
    wr_tbl = all_metrics.pivot_table(index='pattern', columns='split', values='win_rate', aggfunc='first')
    wr_tbl = wr_tbl.reindex(columns=['train','val','test'])
    x = np.arange(len(wr_tbl))
    w = 0.25
    for k, (split, color) in enumerate(SPLIT_COLORS.items()):
        if split in wr_tbl.columns:
            ax1.bar(x + k*w, wr_tbl[split].fillna(0), w, label=split, color=color, alpha=0.85)
    ax1.set_xticks(x + w)
    ax1.set_xticklabels(wr_tbl.index, rotation=30, fontsize=7)
    ax1.axhline(0.5, color='white', ls='--', lw=0.8)
    ax1.set_ylim(0, 1)
    ax1.legend(fontsize=8)
ax1.set_title('Win Rate: Train / Val / Test')

# 2. Expected PnL 对比
ax2 = fig.add_subplot(gs[0, 1])
if not all_metrics.empty:
    exp_tbl = all_metrics.pivot_table(index='pattern', columns='split', values='expect_pct', aggfunc='first')
    exp_tbl = exp_tbl.reindex(columns=['train','val','test'])
    for k, (split, color) in enumerate(SPLIT_COLORS.items()):
        if split in exp_tbl.columns:
            ax2.bar(x + k*w, exp_tbl[split].fillna(0), w, label=split, color=color, alpha=0.85)
    ax2.axhline(0, color='white', ls='--', lw=0.8)
    ax2.set_xticks(x + w)
    ax2.set_xticklabels(exp_tbl.index, rotation=30, fontsize=7)
    ax2.legend(fontsize=8)
ax2.set_title('Expected P&L %: Train / Val / Test')

# 3. 样本量对比
ax3 = fig.add_subplot(gs[0, 2])
if not all_metrics.empty:
    n_tbl = all_metrics.pivot_table(index='pattern', columns='split', values='n_trades', aggfunc='first')
    n_tbl = n_tbl.reindex(columns=['train','val','test'])
    for k, (split, color) in enumerate(SPLIT_COLORS.items()):
        if split in n_tbl.columns:
            ax3.bar(x + k*w, n_tbl[split].fillna(0), w, label=split, color=color, alpha=0.85)
    ax3.axhline(MIN_SAMPLES, color='orange', ls='--', lw=1.0, label=f'Min={MIN_SAMPLES}')
    ax3.set_xticks(x + w)
    ax3.set_xticklabels(n_tbl.index, rotation=30, fontsize=7)
    ax3.legend(fontsize=8)
ax3.set_title(f'Sample Count (Min={MIN_SAMPLES})')

# 4. 置信度校准 (Train+Val)
ax4 = fig.add_subplot(gs[1, 0])
if not trainval_trades.empty:
    mid = (calib_tv['bin_low'] + calib_tv['bin_high']) / 2
    ax4.plot([mid.min(), mid.max()], [mid.min(), mid.max()],
             'w--', lw=0.8, label='Perfect')
    colors_cal = ['#26a69a' if s else 'gray' for s in calib_tv['sufficient']]
    ax4.scatter(mid, calib_tv['actual_win_rate'], c=colors_cal,
                s=[max(30, min(120, n*4)) for n in calib_tv['n_trades']], zorder=5)
    for i, row in calib_tv.iterrows():
        ax4.annotate(f"n={row['n_trades']}", (mid[i], row['actual_win_rate']),
                     fontsize=7, ha='center', va='bottom', color='white')
    ax4.set_xlabel('Model Confidence', color='white')
    ax4.set_ylabel('Actual Win Rate', color='white')
    ax4.legend(fontsize=8)
    ax4.set_facecolor('#111')
ax4.set_title('Confidence Calibration (Train+Val)')

# 5. 学习曲线 (全数据)
ax5 = fig.add_subplot(gs[1, 1])
all_trades_ts = pd.concat([results[s]['trades'] for s in ['train','val','test']
                            if not results[s]['trades'].empty], ignore_index=True)
if not all_trades_ts.empty:
    lc = learning_curve(all_trades_ts, n_blocks=7)
    bar_colors = ['#26a69a' if v >= 0.5 else '#ef5350' for v in lc['win_rate']]
    ax5.bar(range(len(lc)), lc['win_rate'], color=bar_colors, alpha=0.85)
    ax5.axhline(0.5, color='white', ls='--', lw=0.8)
    ax5.set_xticks(range(len(lc)))
    ax5.set_xticklabels([str(b)[:10] for b in lc['block_start']], rotation=25, fontsize=7)
    ax5.set_ylim(0, 1)
    ax5.set_ylabel('Win Rate', color='white')
    # 垂直线标记 train/val/test 边界
    for blk_i, blk_start in enumerate(lc['block_start']):
        if str(blk_start) >= str(val_dates[0].date())[:10] and blk_i > 0:
            ax5.axvline(blk_i - 0.5, color='#2196F3', ls=':', lw=1.2, label='Val start' if blk_i < 3 else '')
            break
    for blk_i, blk_start in enumerate(lc['block_start']):
        if str(blk_start) >= str(test_dates[0].date())[:10] and blk_i > 0:
            ax5.axvline(blk_i - 0.5, color='#FF5722', ls=':', lw=1.2, label='Test start')
            break
    ax5.legend(fontsize=8)
ax5.set_title('Win Rate Stability (Learning Curve)')

# 6. 累积 PnL — Train vs Test
ax6 = fig.add_subplot(gs[1, 2])
for split_name, color in [('train', '#4CAF50'), ('test', '#FF5722')]:
    tr = results[split_name]['trades']
    if not tr.empty:
        cum = tr.sort_values('entry_time')['pnl_pct'].cumsum().reset_index(drop=True)
        ax6.plot(range(len(cum)), cum, label=split_name, color=color, lw=1.5)
ax6.axhline(0, color='white', ls='--', lw=0.5)
ax6.legend(fontsize=9)
ax6.set_xlabel('Trade #', color='white')
ax6.set_ylabel('Cumulative P&L %', color='white')
ax6.set_title('Cumulative P&L: Train vs Test')

# 7. Confidence 阈值扫描 (Train+Val)
ax7 = fig.add_subplot(gs[2, 0])
if not trainval_trades.empty:
    thresholds = np.arange(0.50, 0.82, 0.02)
    sweep_wr, sweep_n = [], []
    for thresh in thresholds:
        sub = trainval_trades[trainval_trades['confidence'] >= thresh]
        sweep_wr.append(float((sub['pnl_pct'] > 0).mean()) if not sub.empty else np.nan)
        sweep_n.append(len(sub))
    ax7_twin = ax7.twinx()
    ax7.plot(thresholds, sweep_wr, color='#26a69a', lw=2, marker='o', ms=4, label='Win Rate')
    ax7_twin.bar(thresholds, sweep_n, width=0.015, alpha=0.3, color='steelblue', label='n_trades')
    ax7.axhline(0.5, color='white', ls='--', lw=0.8)
    ax7.set_xlabel('Min Confidence', color='white')
    ax7.set_ylabel('Win Rate', color='#26a69a')
    ax7_twin.set_ylabel('n_trades', color='steelblue')
    ax7.set_ylim(0, 1)
    ax7.legend(loc='upper left', fontsize=8)
ax7.set_title('Confidence Threshold Sweep (Train+Val)')

# 8. 调参建议摘要
ax8 = fig.add_subplot(gs[2, 1])
ax8.axis('off')
suggestions = [
    'Param Tuning Candidates (Train+Val only):',
    '',
    '① orb_breakout.rvol_threshold',
    '  1.3 → 1.5  (reduce signals, ↑WR)',
    '',
    '② last_hour_drift.trigger_after_minute',
    '  300 → 330  (later entry, stronger drift)',
    '',
    '③ vwap_rejection.prior_bars_one_side',
    '  5 → 3  (more signals, need WR check)',
    '',
    '⚠️ None applied yet.',
    '  Need ≥60 days to validate.',
]
ax8.text(0.05, 0.95, '\n'.join(suggestions), transform=ax8.transAxes,
         fontsize=8, verticalalignment='top', color='white',
         bbox=dict(boxstyle='round', facecolor='#222', alpha=0.8))
ax8.set_title('Tuning Notes')

# 9. 最终 Test 集 per-pattern 胜率
ax9 = fig.add_subplot(gs[2, 2])
if not test_trades.empty:
    test_wr = test_trades.groupby('pattern').apply(
        lambda g: pd.Series({'win_rate': (g['pnl_pct'] > 0).mean(), 'n': len(g)})
    )
    bar_c = ['#26a69a' if w >= 0.5 else '#ef5350' for w in test_wr['win_rate']]
    ax9.bar(range(len(test_wr)), test_wr['win_rate'], color=bar_c, alpha=0.85)
    for i, (pat, row) in enumerate(test_wr.iterrows()):
        suf = '✅' if row['n'] >= MIN_SAMPLES else '⚠️'
        ax9.text(i, row['win_rate'] + 0.02, f"{suf}\nn={int(row['n'])}",
                 ha='center', fontsize=7, color='white')
    ax9.axhline(0.5, color='white', ls='--', lw=0.8)
    ax9.set_xticks(range(len(test_wr)))
    ax9.set_xticklabels(test_wr.index, rotation=30, fontsize=7)
    ax9.set_ylim(0, 1.1)
ax9.set_title('Test Set Win Rate (Final OOS)')

for ax in fig.get_axes():
    ax.set_facecolor('#1a1a2e')
    ax.tick_params(colors='white')
    ax.title.set_color('white')

fig.patch.set_facecolor('#0f0f1a')
plt.savefig('M7_oos_validation.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.close()
print('Saved M7_oos_validation.png')

Saved M7_oos_validation.png


## M7 OOS 结论

### 方法论
- **时间切分** (无数据泄漏): Train → Val → Test，严格按时间顺序
- **调参范围**: 仅 Train+Val，Test 集一次性最终评估
- **有效性标准**: Test 集每个 pattern ≥ 30 样本 + 二项检验 p < 0.05

### 当前数据局限
| 指标 | 现状 |
|---|---|
| 总数据 | 14 个交易日 |
| Test 集 | ~2 个交易日 |
| 统计有效 pattern | 0 (样本量不足) |
| 建议最低数据 | ≥ 60 个交易日 (~3个月) |

### 调参纪律
- 所有参数变更记录于 `config/changelog.md`
- 现阶段参数维持 v0.1.0 初始设定，待数据积累后再评估

### 下一步
1. 扩充历史数据 (yfinance 或 Polygon.io)
2. 积累 ≥ 60 天后重跑本 notebook
3. 基于 Train+Val 调整参数，记录 changelog
4. 最终一次性 Test 集评估

---
**M1–M7 全部完成。** 项目基础版本 v0.1.0 Ready.